# Day 4 — Feature Selection
**Week 3 · Feature Engineering & Data Prep · AI Engineer Lab**

More features ≠ better models. This notebook covers:
- Variance threshold (remove near-constant features)
- Correlation-based filtering (remove redundant features)
- Univariate selection (f-statistic, mutual information)
- Recursive Feature Elimination (RFE / RFECV)
- Embedded selection via tree importance and L1 regularisation
- SHAP values for model-agnostic importance
- Feature selection inside a Pipeline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_classif,
    mutual_info_classif, RFE, RFECV, SelectFromModel
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.datasets import make_classification

np.random.seed(42)
print('Libraries loaded.')

## 1. Create a High-Dimensional Dataset with Known Relevant Features

In [ ]:
# 50 features: 10 informative, 10 redundant (correlated), 30 noise
X, y = make_classification(
    n_samples=2000,
    n_features=50,
    n_informative=10,
    n_redundant=10,
    n_repeated=0,
    n_classes=2,
    random_state=42,
    shuffle=False
)

# Add near-constant features (should be removed by VarianceThreshold)
near_const = np.random.choice([0, 1], size=(2000, 5), p=[0.99, 0.01])
constant = np.zeros((2000, 3))  # 100% constant
X = np.hstack([X, near_const, constant])

feature_names = ([f'informative_{i}' for i in range(10)] +
                 [f'redundant_{i}' for i in range(10)] +
                 [f'noise_{i}' for i in range(30)] +
                 [f'near_const_{i}' for i in range(5)] +
                 [f'constant_{i}' for i in range(3)])

df = pd.DataFrame(X, columns=feature_names)
print(f'Dataset: {df.shape[0]} samples, {df.shape[1]} features')
print(f'Classes: {np.bincount(y)}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Variance Threshold — Remove Near-Constant Features

In [ ]:
# All constant features have variance=0; near-constant have variance < threshold
vt = VarianceThreshold(threshold=0.01)
X_train_vt = vt.fit_transform(X_train)
X_test_vt  = vt.transform(X_test)

removed_cols = [feature_names[i] for i in range(len(feature_names)) if not vt.get_support()[i]]
print(f'Features before: {X_train.shape[1]}')
print(f'Features after VarianceThreshold(0.01): {X_train_vt.shape[1]}')
print(f'Removed features: {removed_cols}')

# Visualise variances
variances = pd.Series(vt.variances_, index=feature_names)
low_var = variances.nsmallest(15)

plt.figure(figsize=(10, 4))
low_var.plot(kind='barh', color=['#ef4444' if v < 0.01 else '#3b82f6' for v in low_var])
plt.axvline(x=0.01, color='orange', linestyle='--', label='threshold=0.01')
plt.title('15 Lowest-Variance Features')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Correlation-Based Filtering — Remove Redundant Features

In [ ]:
def remove_correlated_features(X, feature_names, threshold=0.90):
    """Remove one of each pair of features with correlation > threshold."""
    df_tmp = pd.DataFrame(X, columns=feature_names)
    corr = df_tmp.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return to_drop

# Apply to variance-filtered data
remaining_features = [feature_names[i] for i in range(len(feature_names)) if vt.get_support()[i]]
to_drop = remove_correlated_features(X_train_vt, remaining_features, threshold=0.90)

print(f'Features with high correlation (>0.90): {len(to_drop)}')
print(f'Dropped: {to_drop[:10]}...')

# Show correlation heatmap of informative vs redundant features
sample_feats = [f for f in remaining_features if 'informative' in f or 'redundant' in f][:10]
corr_sub = pd.DataFrame(X_train_vt, columns=remaining_features)[sample_feats].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_sub, dtype=bool))
sns.heatmap(corr_sub, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.3, annot_kws={'size': 8})
plt.title('Correlation Matrix — Informative vs Redundant Features')
plt.tight_layout()
plt.show()

## 4. Univariate Feature Selection

In [ ]:
# F-statistic (ANOVA): linear relationship between feature and target
sel_f = SelectKBest(f_classif, k=15)
X_train_f = sel_f.fit_transform(X_train, y_train)
f_scores = pd.Series(sel_f.scores_, index=feature_names).sort_values(ascending=False)

# Mutual Information: captures non-linear dependencies
sel_mi = SelectKBest(mutual_info_classif, k=15)
X_train_mi = sel_mi.fit_transform(X_train, y_train)
mi_scores = pd.Series(sel_mi.scores_, index=feature_names).sort_values(ascending=False)

# Compare top-15 features selected by each
f_top15 = set(f_scores.nlargest(15).index)
mi_top15 = set(mi_scores.nlargest(15).index)
agreement = f_top15 & mi_top15
print(f'F-statistic top 15: {sorted(f_top15)[:5]}...')
print(f'Mutual Info top 15: {sorted(mi_top15)[:5]}...')
print(f'Agreement (both selected): {sorted(agreement)}')

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
f_scores.head(20).plot(kind='bar', ax=axes[0], color='#3b82f6', alpha=0.8)
axes[0].set_title('F-Statistic Scores (top 20)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

mi_scores.head(20).plot(kind='bar', ax=axes[1], color='#a78bfa', alpha=0.8)
axes[1].set_title('Mutual Information Scores (top 20)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Recursive Feature Elimination with Cross-Validation (RFECV)

In [ ]:
# RFECV automatically finds the optimal number of features
rfecv = RFECV(
    estimator=LogisticRegression(max_iter=1000),
    step=1,
    cv=3,
    scoring='roc_auc',
    min_features_to_select=5,
    n_jobs=-1
)

# Use scaled data for logistic regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)

rfecv.fit(X_train_sc, y_train)
print(f'Optimal number of features: {rfecv.n_features_}')
selected = [feature_names[i] for i in range(len(feature_names)) if rfecv.support_[i]]
print(f'Selected features: {selected[:10]}...')

# Plot CV score vs number of features
plt.figure(figsize=(10, 4))
n_features_range = range(rfecv.min_features_to_select,
                          len(rfecv.cv_results_['mean_test_score']) + rfecv.min_features_to_select)
mean_scores = rfecv.cv_results_['mean_test_score']
std_scores = rfecv.cv_results_['std_test_score']

plt.plot(n_features_range, mean_scores, color='#3b82f6', linewidth=2)
plt.fill_between(n_features_range,
                  mean_scores - std_scores,
                  mean_scores + std_scores,
                  alpha=0.15, color='#3b82f6')
plt.axvline(x=rfecv.n_features_, color='#f59e0b', linestyle='--',
            label=f'Optimal: {rfecv.n_features_} features')
plt.xlabel('Number of Features')
plt.ylabel('CV AUC')
plt.title('RFECV: CV Score vs Number of Features')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Embedded Selection — Tree Feature Importance

In [ ]:
# Random Forest — feature importances from mean decrease in impurity
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=feature_names)
importances_sorted = importances.sort_values(ascending=False)

plt.figure(figsize=(12, 5))
colors = ['#22c55e' if 'informative' in f else '#f59e0b' if 'redundant' in f
          else '#ef4444' for f in importances_sorted.head(25).index]
importances_sorted.head(25).plot(kind='bar', color=colors, alpha=0.85)
plt.title('Random Forest Feature Importances (Top 25)\nGreen=informative, Yellow=redundant, Red=noise')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# SelectFromModel: keep features above threshold
sfm = SelectFromModel(rf, threshold='mean', prefit=True)
X_train_sfm = sfm.transform(X_train)
print(f'Features after SelectFromModel (threshold=mean): {X_train_sfm.shape[1]}')

## 7. L1 Regularisation (Lasso) for Feature Selection

In [ ]:
# L1 drives weak feature weights to exactly 0
# Use logistic regression with L1 penalty
l1_model = LogisticRegression(penalty='l1', C=0.1, solver='saga', max_iter=2000, random_state=42)
l1_model.fit(X_train_sc, y_train)

coefs = pd.Series(np.abs(l1_model.coef_[0]), index=feature_names)
n_nonzero = (coefs > 0).sum()
print(f'Non-zero coefficients (selected features): {n_nonzero} / {len(feature_names)}')

# Selected vs zeroed features
lasso_selected = coefs[coefs > 0].sort_values(ascending=False)
print('\nL1-selected features (non-zero weights):')
print(lasso_selected.head(15))

# Show that informative features survive, noise features are zeroed
info_selected = sum(1 for f in lasso_selected.index if 'informative' in f)
noise_selected = sum(1 for f in lasso_selected.index if 'noise' in f)
print(f'\nInformative features surviving L1: {info_selected}/10')
print(f'Noise features surviving L1: {noise_selected}/30')

## 8. Compare All Selection Methods

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

strategies = {
    'All features (58)': X,
    'Variance threshold': vt.transform(X),
    'SelectKBest MI (15)': sel_mi.transform(X),
    'RFECV optimal': rfecv.transform(scaler.transform(X)),
    'Tree SelectFromModel': sfm.transform(X),
}

print('=== Feature Selection Comparison (5-Fold AUC, Logistic Regression) ===')
for name, X_sel in strategies.items():
    # Use scaled features for logistic regression
    pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))])
    cv = cross_val_score(pipe, X_sel, y, cv=5, scoring='roc_auc')
    print(f'  {name:<35} features={X_sel.shape[1]:<4} AUC={cv.mean():.4f} ± {cv.std():.4f}')

## Exercises

1. **Permutation importance**: Use `sklearn.inspection.permutation_importance` to calculate feature importance for the Random Forest. Compare to the built-in `feature_importances_`. Which is more reliable for correlated features?

2. **Lasso regularisation path**: Fit Lasso with multiple values of alpha (0.001 to 1.0). Plot the number of non-zero features vs alpha. Identify the alpha that retains only the 10 informative features.

3. **Feature selection inside CV**: Build a pipeline with `SelectKBest(mutual_info_classif, k=15) → StandardScaler → LogisticRegression`. Run 5-fold CV. Is the AUC better than selecting features outside CV first?

4. **Curse of dimensionality**: Demonstrate KNN's performance degradation as you add noise features to the dataset. Plot AUC vs number of features (from 10 informative features up to 58 total).

In [ ]:
# ── Exercise 4 Solution: Curse of dimensionality ──
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
# Create versions with increasing noise features
feature_counts = [10, 15, 20, 30, 40, 50, 58]
aucs = []

for n_feat in feature_counts:
    X_sub = X[:, :n_feat]  # take first n_feat columns (informative are first 10)
    pipe = Pipeline([('sc', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
    cv = cross_val_score(pipe, X_sub, y, cv=5, scoring='roc_auc')
    aucs.append(cv.mean())

plt.figure(figsize=(8, 4))
plt.plot(feature_counts, aucs, 'o-', color='#3b82f6', linewidth=2)
plt.axvline(x=10, color='#22c55e', linestyle='--', label='10 informative features')
plt.xlabel('Number of Features')
plt.ylabel('CV AUC')
plt.title('KNN: Performance vs Number of Features (Curse of Dimensionality)')
plt.legend()
plt.tight_layout()
plt.show()
print('AUC decreases as noise features are added — KNN suffers most from high dimensionality.')